# REN–HH 정책군의 가구별 부담과 환류
거시 시나리오 BAU·TOU·TOU_REN_HH를 읽고 HH 몫을 U·I·IC로 배분합니다.
C 단독은 보조 분석으로 계산하되 본문 표에서 분리합니다.
미시 U/I/C/IC는 별도 CGE 시나리오가 아닙니다. 각 정책점의 동일한 가격과 HH 예산을 사용합니다.
기존 공통 표본·균등화소득 정의를 유지하며 실제 IBT–TOU 청구식 노트북과 구분합니다.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Run_Master.gms').is_file())
ROOT=PROJECT/'Micro'
sys.path.insert(0,str(ROOT))
from micro_common import load_config,prepare_common_sample,export_common_sample,profile_tag
from tou_policy_results import load_policy_results
CONFIG=load_config(ROOT/'micro_common_config.json')
GDX_FILE=PROJECT/'TOU_recycling_results.gdx'
GAMS_DIR=Path(r'C:\GAMS\40')
REGION='01_KOR'
SELECTED_RATE='r100'
SELECTED_MIX='a50'
SHIFT_REALIZATION=0.0  # 0: 소비패턴 고정. 0~1: A8까지 가능한 peak→off 이전 실현 비율
INCLUDE_C_APPENDIX=True
OUTPUT_DIR=ROOT/'policy_analysis'/('micro_'+profile_tag(CONFIG))
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
d,h,sample_flow,income_summary,common_meta=prepare_common_sample(ROOT/'survey_data.xlsx',CONFIG)
export_common_sample(ROOT,CONFIG,d,h,sample_flow,income_summary,common_meta)
x=load_policy_results(GDX_FILE,GAMS_DIR)
BLOCKS=['peak','mid','off']
display(sample_flow)
display(income_summary)


## 1. CGE 가격과 가계 예산 연결
β_HH=각 정책점의 HH 예산/CGE의 BAU 가계 전력지출.
표본 월 HH 예산=β_HH×Σ(가중치×기준 전력량요금, 세금 제외).
원화 환율·전국 가구 수를 임의로 넣지 않는 표본 등가 예산입니다.
REN 예산을 가계에 다시 배분하지 않습니다. REN의 가격 효과는 CGE 시간대 가격배율에 이미 포함됩니다.
기존 기본요금·기후/연료비는 유지한 비례 가격 전가입니다. 직접 TOU 청구식 효과를 다시 더하지 않습니다.


In [ ]:
base_prices=x['prices_base'].loc[lambda t:t.region.eq(REGION)].pivot(index='scenario',columns='block',values='value')
policy_prices=x['prices_policy'].loc[lambda t:t.region.eq(REGION)].pivot(index=['rate','mix'],columns='block',values='value')
links=x['budgets'].loc[lambda t:t.region.eq(REGION)].copy()
macro_energy=float(x['bau_electricity_expenditure'].loc[REGION])
sample_energy=float(np.dot(h.weight,h.energy_net_monthly))
links['sample_hh_budget_monthly_won']=links.budget_hh/macro_energy*sample_energy
links['sample_total_budget_equivalent_won']=links.budget_total/macro_energy*sample_energy
display(links.loc[links.rate.eq(SELECTED_RATE)&links.mix.isin(['a00','a50','a100'])][
    ['policy_id','scenario','budget_hh','budget_ren','sample_hh_budget_monthly_won']])


## 2. 배분 규칙과 부담 지표
U=모든 가구 동일액, I=균등화소득 저소득 분위에 동일액, IC=저소득 여부×제약 점수.
C=제약 점수만 사용한 보조 결과. 점수는 A8/최초 peak비중이며 peak=0이면 0입니다.
이 배분은 조정 제약 자체를 바꾸지 않고 불균등한 부담을 보상합니다.
동일 소득 동률을 묶으므로 저소득 Q1·Q2가 정확히 하위 40%는 아닐 수 있습니다.


In [ ]:
w=h.weight.to_numpy()
original_shares=h[['s_'+b for b in BLOCKS]].to_numpy()
shifted=original_shares.copy()
cap=(h.s_peak-h.peak_min_share).to_numpy()
shifted[:,0]-=SHIFT_REALIZATION*cap
shifted[:,2]+=SHIFT_REALIZATION*cap
allocations=['U','I','IC']+(['C'] if INCLUDE_C_APPENDIX else [])
scores={a:h['score_TOU_'+a].to_numpy(dtype=float) for a in allocations}
score_totals={a:float(np.dot(w,s)) for a,s in scores.items()}
keep=['household_id','weight','income_quintile','low_income','constraint_score',
      'income_monthly_won','eq_income_monthly_won','kwh']

def household_result(policy_id,scenario,rate,mix,r,a,ratio,budget,allocation):
    result=h[keep].copy()
    share=original_shares if scenario=='BAU' else shifted
    bill=h.a1+h.energy_gross_monthly*(share@ratio-1)
    transfer=np.zeros(len(h)) if allocation=='none' else budget*scores[allocation]/score_totals[allocation]
    result['policy_id']=policy_id;result['scenario']=scenario
    result['rate']=rate;result['mix']=mix
    result['recycling_fraction']=r;result['ren_fraction']=a
    result['allocation']=allocation
    result['analysis_role']='appendix' if allocation=='C' else 'main'
    result['bill_bau_monthly_won']=h.a1
    result['gross_bill_monthly_won']=bill
    result['rebate_monthly_won']=transfer
    result['net_bill_monthly_won']=bill-transfer
    result['gross_change_monthly_won']=bill-h.a1
    result['net_change_monthly_won']=bill-transfer-h.a1
    result['net_change_pct_bau_bill']=result.net_change_monthly_won/h.a1*100
    result['burden_change_pp']=result.net_change_monthly_won/h.income_monthly_won.where(h.income_monthly_won.gt(0))*100
    result['uncompensated_burden_won']=result.net_change_monthly_won.clip(lower=0)
    return result

def mean(g,column):
    valid=g[column].notna()
    return np.average(g.loc[valid,column],weights=g.loc[valid,'weight']) if valid.any() else np.nan

def summarize(g):
    first=g.iloc[0]
    result={c:first[c] for c in ['policy_id','scenario','rate','mix','recycling_fraction','ren_fraction','allocation','analysis_role']}
    result.update(households=len(g),weighted_households=g.weight.sum(),
        gross_change_won=mean(g,'gross_change_monthly_won'),rebate_won=mean(g,'rebate_monthly_won'),
        net_change_won=mean(g,'net_change_monthly_won'),mean_bill_change_pct=mean(g,'net_change_pct_bau_bill'),
        mean_income_burden_change_pp=mean(g,'burden_change_pp'),
        uncompensated_burden_won=mean(g,'uncompensated_burden_won'),
        burden_increase_share=np.average(g.net_change_monthly_won.gt(1e-6),weights=g.weight))
    return result


## 3. 전체 정책 격자 계산
전체 격자는 요약표로 저장하고 가구별 상세자료는 선택한 (r,a)와 BAU·TOU만 저장합니다.
r=0 중복은 TOU 한 번으로, HH 몫이 0인 REN은 배분 방식 none 한 번으로 처리합니다.
기존 큰 가구별 파일을 반복 생성하지 않도록 현재 선택점의 상세자료만 보존합니다.


In [ ]:
summary_rows={'overall':[],'quintile':[],'low_income':[],'constraint':[],'joint':[]}
selected_households=[]
def collect(frame,save_households=False):
    summary_rows['overall'].append(summarize(frame))
    groups=[('quintile','income_quintile',frame.income_quintile),
            ('low_income','low_income',frame.low_income),
            ('constraint','high_constraint',frame.constraint_score.ge(.5))]
    for name,col,values in groups:
        for value,ix in values.groupby(values).groups.items():
            g=frame.loc[ix]
            summary_rows[name].append(dict(summarize(g),**{col:value}))
    joint=frame.assign(high_constraint=frame.constraint_score.ge(.5))
    for (low,high),g in joint.groupby(['low_income','high_constraint']):
        summary_rows['joint'].append(dict(summarize(g),low_income=low,high_constraint=high))
    if save_households:selected_households.append(frame)
for scenario in ['BAU','TOU']:
    frame=household_result(scenario,scenario,'base','base',0,np.nan,
        base_prices.loc[scenario,BLOCKS].to_numpy(),0,'none')
    collect(frame,True)
for point in links.loc[links.recycling_fraction.gt(0)].itertuples(index=False):
    ratio=policy_prices.loc[(point.rate,point.mix),BLOCKS].to_numpy()
    choices=allocations if point.budget_hh>0 else ['none']
    for allocation in choices:
        frame=household_result(point.policy_id,point.scenario,point.rate,point.mix,
            point.recycling_fraction,point.ren_fraction,ratio,point.sample_hh_budget_monthly_won,allocation)
        collect(frame,point.rate==SELECTED_RATE and point.mix==SELECTED_MIX)
summary_tables={name:pd.DataFrame(rows) for name,rows in summary_rows.items()}
household_selected=pd.concat(selected_households,ignore_index=True)
overall=summary_tables['overall']
view=overall.loc[overall.analysis_role.eq('main')&(
    overall.policy_id.isin(['BAU','TOU'])|(overall.rate.eq(SELECTED_RATE)&overall.mix.eq(SELECTED_MIX)))]
display(view[['policy_id','scenario','allocation','gross_change_won','rebate_won','net_change_won','burden_increase_share']])


## 4. 소득·제약별 보호 성과와 환류 규모
순부담은 전기요금 증가−현금 환류입니다. 음수는 가계 순지출 감소이며 음의 전력가격을 뜻하지 않습니다.
소득 대비 부담은 실제 가구 월소득을 분모로 사용하고, 소득 0인 가구는 해당 지표에서만 제외합니다.
REN 단독은 가계 현금 환류가 없어 모든 배분 방식에 동일합니다.
미시 결과는 조건부 직접 전기요금 효과이며 대표가계 EV와 같은 지표가 아닙니다.


In [ ]:
low=summary_tables['low_income']
display(low.loc[low.low_income & low.analysis_role.eq('main') & low.rate.eq(SELECTED_RATE)&
    low.mix.isin(['a00',SELECTED_MIX,'a100'])][['policy_id','scenario','allocation','net_change_won','burden_increase_share','uncompensated_burden_won']])
q=summary_tables['quintile']
display(q.loc[q.analysis_role.eq('main')&q.rate.eq(SELECTED_RATE)&q.mix.eq(SELECTED_MIX)].pivot(
    index='income_quintile',columns='allocation',values='net_change_won'))


## 5. 저장과 분석 범위
설정의 선택점 상세자료와 전체 격자의 요약자료를 분리 저장합니다.
직접 IBT–TOU 청구식 노트북은 유지합니다. GDX는 기존 승인 요금 입력이며 공통 표본 기타계절 후보와 다를 수 있습니다.
이 노트북의 가격배율 전가 결과와 직접 청구식 결과의 차이를 일반균형 효과만으로 해석하지 않습니다.


In [ ]:
for name,table in summary_tables.items():
    table.loc[table.analysis_role.eq('main')].to_csv(OUTPUT_DIR/(name+'_summary.csv'),index=False,encoding='utf-8-sig')
    table.loc[table.analysis_role.eq('appendix')].to_csv(OUTPUT_DIR/('appendix_C_'+name+'.csv'),index=False,encoding='utf-8-sig')
household_selected.to_csv(OUTPUT_DIR/'household_selected.csv',index=False,encoding='utf-8-sig')
links.to_csv(OUTPUT_DIR/'macro_micro_link.csv',index=False,encoding='utf-8-sig')
income_summary.to_csv(OUTPUT_DIR/'income_reference_comparison.csv',index=False,encoding='utf-8-sig')
metadata=dict(common_sample=common_meta,gdx_file=str(GDX_FILE),gdx_modified=GDX_FILE.stat().st_mtime,
    selected_rate=SELECTED_RATE,selected_mix=SELECTED_MIX,shift_realization=SHIFT_REALIZATION,
    primary_allocations=['U','I','IC'],C_appendix=INCLUDE_C_APPENDIX,
    budget_method='HH share only / CGE BAU electricity expenditure * sample BAU net energy charges',
    price_method='CGE block price ratios applied proportionally; REN effect included once',
    scope='Conditional household bill incidence; not national totals or household EV')
(OUTPUT_DIR/'definitions.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('공통 분석 가구:',len(h),'선택점 상세 행:',len(household_selected))
print('저장 위치:',OUTPUT_DIR)
